# 12 — Ablation studies

**Purpose.** Effect of each protective layer (secure aggregation, clipping, robust median) on accuracy, with and without a sign-flip adversary.

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/12_ablation_studies_*.png|pdf` and `reports/notebooks/12_ablation_studies.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "12_ablation_studies"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Ablate

In [2]:
from federated_learning.synthetic_data import make_dataset
from federated_learning.data_partitioning import partition
from federated_learning.servers.orchestrator import run_fl
from federated_learning.training import FLConfig
from federated_learning import attacks as A
from federated_learning.aggregators import WeightedMean, CoordinateMedian, ClippedMean
from federated_learning.privacy.secure_aggregation import SecAggAggregator
d = make_dataset(n_samples=6000, seed=SEED); parts = partition("iid", d.y_train, 10, SEED); cfg = FLConfig(rounds=10, local_epochs=1, seed=SEED); hook = A.byzantine_hook("sign_flip", {0, 1}, 5.0)
cfgs = {"plain mean": WeightedMean, "+ secure aggregation": SecAggAggregator, "+ clipping": lambda: ClippedMean(1.0), "+ coordinate median": CoordinateMedian}
rows = []
for k, f in cfgs.items():
    rows.append(dict(configuration=k, clean_acc=run_fl(cfg, d, parts, aggregator=f()).final["acc"], sign_flip_acc=run_fl(cfg, d, parts, aggregator=f(), client_hook=hook).final["acc"]))
tab = pd.DataFrame(rows).round(4); display(tab); SUMMARY["ablation"] = tab.to_dict("records")
print("Secure aggregation hides individual updates but cannot detect a poisoned one: confidentiality and integrity need different components.")

,configuration,clean_acc,sign_flip_acc
0,plain mean,0.7756,0.3939
1,+ secure aggregation,0.7756,0.3939
2,+ clipping,0.7756,0.4183
3,+ coordinate median,0.7733,0.7650


Secure aggregation hides individual updates but cannot detect a poisoned one: confidentiality and integrity need different components.


## 2. Figure

In [3]:
fig, ax = plt.subplots(figsize=(5.6, 2.9)); x = np.arange(len(tab)); ax.bar(x - .2, tab.clean_acc, .4, label="clean"); ax.bar(x + .2, tab.sign_flip_acc, .4, label="20% sign-flip")
ax.set_xticks(x, tab.configuration, rotation=15); ax.set_ylabel("accuracy"); ax.legend(); save_fig(fig, "ablation"); save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/12_ablation_studies.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.